# Unit 10 - Analysis (Demo)

**Atoms served:** `U10-A1` (match test to metric), `U10-A2` (confidence intervals), `U10-A4` (practical vs statistical significance)

**Estimated runtime:** ~25 seconds

**After this notebook you can:** run a t-test with a confidence interval, bootstrap a skewed metric, analyse a ratio metric correctly, and separate statistical from business significance.

## Without code

1. T-test section: CI should exclude 0 if p < 0.05 for the simulated lift.
2. Bootstrap: CI width similar order to t-test on means.
3. Ratio metric: the mean of per-user `CTR`s and the ratio of sums disagree sharply, and the **lift** they report differs by several times over.
4. Practical significance: the lift is statistically significant **and** sits below the $5 ship threshold, so the honest answer is "real, but not worth building."

`V28` uses critical values; this notebook also reports p-values to bridge unit 08 vocabulary.

## 1. The question

Treatment raised average order value in a simulated checkout test. **Is it statistically detectable? Is it large enough to ship?** Those are different questions (`V29`).

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Continuous revenue per order (lognormal), plus clicks and impressions for a ratio `CTR` metric.

In [ ]:
n = 2000
D = np.repeat([0,1], n//2)
revenue_control = np.random.lognormal(mean=3.4, sigma=0.5, size=n//2)
revenue_treat = np.random.lognormal(mean=3.45, sigma=0.5, size=n//2)
rev = np.concatenate([revenue_control, revenue_treat])
df = pd.DataFrame({'D_i': D, 'revenue': rev})

# Ratio metric data. Impressions are wildly unequal: one user in ten is a heavy
# user who sees ~2000 impressions and clicks rarely, while the rest see ~20 and
# click often. Treatment only helps the heavy users.
heavy = np.random.binomial(1, 0.10, n)
imp = np.maximum(np.where(heavy==1, np.random.poisson(2000, n), np.random.poisson(20, n)), 1)
base_ctr = np.where(heavy==1, 0.02, 0.15)
treat_gain = np.where((D==1) & (heavy==1), 0.01, 0.0)
clk = np.random.binomial(imp, np.clip(base_ctr + treat_gain, 0, 1))
ratio_df = pd.DataFrame({'D_i': D, 'clicks': clk, 'impressions': imp})

## 4. The naive move

Run a t-test, see p < 0.05, ship.

In [ ]:
ctrl = df[df.D_i==0]['revenue']
trt = df[df.D_i==1]['revenue']
tstat, pval = stats.ttest_ind(trt, ctrl, equal_var=False)
ate = trt.mean() - ctrl.mean()
se = np.sqrt(trt.var()/len(trt) + ctrl.var()/len(ctrl))
ci_low, ci_high = ate - 1.96*se, ate + 1.96*se
print('ATE ($):', round(ate,2))
print(f'95% CI: ({ci_low:.2f}, {ci_high:.2f})')
print('p-value:', round(pval,4))

Statistical significance answers "likely not zero." It does not answer "worth building."

## 5. What actually happens

**Bootstrap for awkward metrics (`U10-A1`).** When the sampling distribution is skewed, resampling still builds a CI.

In [ ]:
def bootstrap_ate(data, treat_col, outcome, n_boot=500):
    ates = []
    for _ in range(n_boot):
        samp = data.sample(len(data), replace=True)
        m = samp.groupby(treat_col)[outcome].mean()
        ates.append(m[1]-m[0])
    return np.percentile(ates, [2.5, 97.5])

boot_ci = bootstrap_ate(df, 'D_i', 'revenue')
print('Bootstrap 95% CI for mean revenue ATE:', np.round(boot_ci,2))

Bootstrap CI lands in the same ballpark as the t-based interval for this mean.

**Ratio metric done wrong, then right.** `CTR` is clicks / impressions - not the average of session CTRs.

In [ ]:
per_user_ctr = ratio_df['clicks'] / ratio_df['impressions']
wrong = per_user_ctr.groupby(ratio_df['D_i']).mean()          # every user counts equally
totals = ratio_df.groupby('D_i')[['clicks', 'impressions']].sum()
right = totals['clicks'] / totals['impressions']              # every impression counts equally
print('Wrong (mean of per-user ratios):', wrong.round(4).to_dict())
print('Right (ratio of sums):', right.round(4).to_dict())
print('Lift wrong:', round(wrong[1]-wrong[0], 4), 'lift right:', round(right[1]-right[0], 4))

Look at the two lifts, not the two levels. The mean of per-user ratios gives a user with 20 impressions the same vote as a user with 2,000, so it almost erases an effect that lives entirely in the heavy users. The ratio of sums keeps it. Neither number is a bug in the data - they answer different questions, and only one of them matches "did clicks per impression go up."

**Practical significance (`U10-A4`, `V29`).** Compare `ATE` to a ship threshold.

In [ ]:
ship_threshold_dollars = 5.0  # finance ships only if mean order value rises by $5
print('Observed ATE ($):', round(ate,2))
print('Ship threshold ($):', ship_threshold_dollars)
print('Statistically significant?', pval < 0.05)
print('Practically significant (ship)?', ate >= ship_threshold_dollars)

## 6. What you do about it

- Match the **test to the metric** - means, proportions, ratios each have a path (`U10-A1`).
- Report **confidence intervals** for action ranges (`U10-A2`).
- Use **bootstrap** when formulas are awkward.
- Apply a **business threshold** after statistical significance (`U10-A4`).

**When this matters less:** Guardrail metrics where any harm blocks ship regardless of significance.

---

**Takeaway:** Statistical significance is gate one. Practical significance is the launch decision. Twyman's law (`V29`) still applies - surprise goes to trust checks first.

**Back to the unit:** [V1](../V1/units/unit-10-analysis-decision-and-ethics/README.md) · [V2](../V2/units/unit-10-analysis-decision-and-ethics/README.md)